# 03 Búsqueda vectorial con TF-IDF

**Nombre:** Harry Guajan  
**Fecha:** 23 de octubre de 2026

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

In [11]:
from collections import Counter
import math
import re
import nltk
import numpy as np
import pandas as pd
from IPython.display import display

try:
    from nltk.corpus import gutenberg
    identificadores = gutenberg.fileids()
    if len(identificadores) != 18:
        raise LookupError("La colección Gutenberg de NLTK no está disponible.")
except LookupError:
    nltk.download("gutenberg", quiet=True)
    from nltk.corpus import gutenberg
    identificadores = gutenberg.fileids()
if len(identificadores) != 18:
    raise RuntimeError(f"Se esperaban 18 documentos de Project Gutenberg y se encontraron {len(identificadores)}.")
documentos = [gutenberg.raw(identificador) for identificador in identificadores]

token_pattern = re.compile(r"(?u)\b\w\w+\b")
def tokenizar(texto):
    return token_pattern.findall(texto.lower())

def titulo(texto, identificador):
    nombres = {
        "austen-emma.txt": "Emma", "austen-persuasion.txt": "Persuasion", "austen-sense.txt": "Sense and Sensibility",
        "bible-kjv.txt": "The King James Bible", "blake-poems.txt": "Poems of William Blake", "bryant-stories.txt": "Stories by William Cullen Bryant",
        "burgess-busterbrown.txt": "The Adventures of Buster Bear", "carroll-alice.txt": "Alice's Adventures in Wonderland",
        "chesterton-ball.txt": "The Ball and the Cross", "chesterton-brown.txt": "The Wisdom of Father Brown", "chesterton-thursday.txt": "The Man Who Was Thursday",
        "edgeworth-parents.txt": "The Parent's Assistant", "melville-moby_dick.txt": "Moby Dick", "milton-paradise.txt": "Paradise Lost",
        "shakespeare-caesar.txt": "The Tragedy of Julius Caesar", "shakespeare-hamlet.txt": "The Tragedy of Hamlet", "shakespeare-macbeth.txt": "The Tragedy of Macbeth", "whitman-leaves.txt": "Leaves of Grass",
    }
    return nombres.get(identificador, identificador)

documentos_tokenizados = [tokenizar(documento) for documento in documentos]
titulos = [titulo(texto, identificador) for texto, identificador in zip(documentos, identificadores)]
longitudes = np.array([len(tokens) for tokens in documentos_tokenizados])
vocabulario = sorted({termino for tokens in documentos_tokenizados for termino in tokens})
indice_terminos = {termino: indice for indice, termino in enumerate(vocabulario)}
tabla_corpus = pd.DataFrame({"Identificador": identificadores, "Título": titulos, "Términos únicos": [len(set(tokens)) for tokens in documentos_tokenizados]})
print(f"Documentos: {len(documentos)} | Vocabulario: {len(vocabulario)}")
print(f"Longitud mínima/media/máxima: {longitudes.min()} / {longitudes.mean():.2f} / {longitudes.max()}")
display(tabla_corpus)


Documentos: 18 | Vocabulario: 42063
Longitud mínima/media/máxima: 6582 / 116375.72 / 812281


,Identificador,Título,Términos únicos
0,austen-emma.txt,Emma,7239
1,austen-persuasion.txt,Persuasion,5755
2,austen-sense.txt,Sense and Sensibility,6315
3,bible-kjv.txt,The King James Bible,12731
4,blake-poems.txt,Poems of William Blake,1507
5,bryant-stories.txt,Stories by William Cullen Bryant,3888
6,burgess-busterbrown.txt,The Adventures of Buster Bear,1525
7,carroll-alice.txt,Alice's Adventures in Wonderland,2559
8,chesterton-ball.txt,The Ball and the Cross,8247
9,chesterton-brown.txt,The Wisdom of Father Brown,7723


### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

In [12]:
def calcular_tf(documentos_tokenizados, vocabulario, indice_terminos):
    matriz = np.zeros((len(documentos_tokenizados), len(vocabulario)), dtype=np.int64)
    for fila, tokens in enumerate(documentos_tokenizados):
        for termino, frecuencia in Counter(tokens).items():
            matriz[fila, indice_terminos[termino]] = frecuencia
    return matriz
def calcular_df(matriz_tf):
    return np.count_nonzero(matriz_tf > 0, axis=0)
def calcular_idf(df, n):
    if np.any(df <= 0): raise ValueError("El DF debe ser positivo.")
    return np.log(n / df) + 1
def calcular_tfidf(matriz_tf, idf):
    return matriz_tf.astype(float) * idf
def similitud_coseno(vector_a, vector_b):
    normas = np.linalg.norm(vector_a) * np.linalg.norm(vector_b)
    return 0.0 if normas == 0 else float(np.dot(vector_a, vector_b) / normas)
def calcular_bm25(consulta, documento, df, k1=1.5, b=0.75):
    frecuencias = Counter(documento)
    media = longitudes.mean()
    puntuacion = 0.0
    for termino in set(consulta):
        if termino not in indice_terminos: continue
        frecuencia_documental = df[indice_terminos[termino]]
        idf = math.log((len(documentos) - frecuencia_documental + 0.5) / (frecuencia_documental + 0.5) + 1)
        frecuencia = frecuencias[termino]
        ajuste = 1 - b + b * len(documento) / media
        puntuacion += idf * frecuencia * (k1 + 1) / (frecuencia + k1 * ajuste)
    return puntuacion
print("Funciones TF, DF, IDF, TF-IDF, coseno y BM25 definidas.")


Funciones TF, DF, IDF, TF-IDF, coseno y BM25 definidas.


### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

In [13]:
matriz_tf = calcular_tf(documentos_tokenizados, vocabulario, indice_terminos)
df = calcular_df(matriz_tf)
idf_manual = calcular_idf(df, len(documentos))
matriz_tfidf_manual = calcular_tfidf(matriz_tf, idf_manual)
print(f"Dimensiones de TF-IDF: {matriz_tfidf_manual.shape}")
display(pd.DataFrame({"Término": vocabulario[:6], "TF": matriz_tf[0, :6], "DF": df[:6], "IDF": idf_manual[:6], "TF-IDF": matriz_tfidf_manual[0, :6]}))


Dimensiones de TF-IDF: (18, 42063)


,Término,TF,DF,IDF,TF-IDF
0,00,0,1,3.890372,0.000000
1,000,2,3,2.791759,5.583519
2,00021053,0,1,3.890372,0.000000
3,00081429,0,1,3.890372,0.000000
4,00482129,0,1,3.890372,0.000000
5,01,0,1,3.890372,0.000000


### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizador = TfidfVectorizer(lowercase=True, token_pattern=r"(?u)\b\w\w+\b", use_idf=True, smooth_idf=False, sublinear_tf=False, norm=None)
matriz_tfidf_sklearn = vectorizador.fit_transform(documentos).toarray()
vocabulario_sklearn = vectorizador.get_feature_names_out()
maxima_diferencia = np.max(np.abs(matriz_tfidf_manual - matriz_tfidf_sklearn))
print(f"¿Mismo vocabulario?: {np.array_equal(vocabulario, vocabulario_sklearn)}")
print(f"Máxima diferencia absoluta: {maxima_diferencia:.12e}")
assert np.array_equal(vocabulario, vocabulario_sklearn)
assert np.allclose(matriz_tfidf_manual, matriz_tfidf_sklearn, atol=1e-12)


¿Mismo vocabulario?: True
Máxima diferencia absoluta: 0.000000000000e+00


### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

In [16]:
consultas = {"aventuras y viajes": "adventure journey sea island travel", "relaciones familiares": "family love marriage children home", "conflicto y poder": "king war battle power people", "naturaleza y poesía": "nature earth flowers night poem"}
from sklearn.metrics.pairwise import cosine_similarity
matrices_consulta = vectorizador.transform(list(consultas.values())).toarray()
vector_consulta, vector_documento = matrices_consulta[0], matriz_tfidf_sklearn[0]
terminos_compartidos = [termino for termino in vocabulario if vector_consulta[indice_terminos[termino]] and vector_documento[indice_terminos[termino]]]
productos_parciales = [vector_consulta[indice_terminos[t]] * vector_documento[indice_terminos[t]] for t in terminos_compartidos]
similitud_manual = similitud_coseno(vector_consulta, vector_documento)
similitud_sklearn = float(cosine_similarity([vector_consulta], [vector_documento])[0, 0])
print(f"Términos compartidos: {terminos_compartidos}")
print(f"Productos parciales: {[round(x, 6) for x in productos_parciales]}")
print(f"Producto punto: {np.dot(vector_consulta, vector_documento):.6f}")
print(f"Normas: {np.linalg.norm(vector_consulta):.6f}, {np.linalg.norm(vector_documento):.6f}")
print(f"Similitud manual / sklearn: {similitud_manual:.12f} / {similitud_sklearn:.12f}")
assert np.isclose(similitud_manual, similitud_sklearn)


Términos compartidos: ['adventure', 'journey', 'sea', 'travel']
Productos parciales: [np.float64(5.042133), np.float64(18.789454), np.float64(17.492145), np.float64(17.200484)]
Producto punto: 58.524216
Normas: 3.550664, 14627.730772
Similitud manual / sklearn: 0.001126805813 / 0.001126805813


### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

In [17]:
def fragmento(texto, terminos, limite=180):
    limpio = " ".join(texto.split())
    posiciones = [limpio.lower().find(t) for t in terminos if t]
    posiciones = [p for p in posiciones if p >= 0]
    inicio = max(min(posiciones) - 60, 0) if posiciones else 0
    return limpio[inicio:inicio + limite] + ("..." if len(limpio) > inicio + limite else "")
for nombre, texto_consulta in consultas.items():
    vector = vectorizador.transform([texto_consulta]).toarray()[0]
    puntuaciones = np.array([similitud_coseno(vector, documento) for documento in matriz_tfidf_sklearn])
    orden = np.argsort(-puntuaciones)[:5]
    print(f"\n## Consulta: {nombre} ({texto_consulta})")
    display(pd.DataFrame([{"Posición": p, "Identificador": identificadores[i], "Título": titulos[i], "Puntuación": round(float(puntuaciones[i]), 6), "Fragmento": fragmento(documentos[i], tokenizar(texto_consulta))} for p, i in enumerate(orden, 1)]))
    puntuaciones_bm25 = np.array([calcular_bm25(tokenizar(texto_consulta), tokens, df) for tokens in documentos_tokenizados])
    orden_bm25 = np.argsort(-puntuaciones_bm25)[:5]
    print("Top-5 BM25")
    display(pd.DataFrame([{"Posición": p, "Identificador": identificadores[i], "Título": titulos[i], "Puntuación BM25": round(float(puntuaciones_bm25[i]), 6)} for p, i in enumerate(orden_bm25, 1)]))



## Consulta: aventuras y viajes (adventure journey sea island travel)


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,melville-moby_dick.txt,Moby Dick,0.010120,rooked serpent; and he shall slay the dragon t...
1,2,whitman-leaves.txt,Leaves of Grass,0.007802,", I above all promote brave soldiers. } In Cab..."
2,3,chesterton-ball.txt,The Ball and the Cross,0.006429,"seemed to be the upper part of a huge, dark o..."
3,4,chesterton-brown.txt,The Wisdom of Father Brown,0.005104,"st and specialist in certain moral disorders, ..."
4,5,milton-paradise.txt,Paradise Lost,0.004749,"n, till one greater Man Restore us, and regain..."


Top-5 BM25


,Posición,Identificador,Título,Puntuación BM25
0,1,whitman-leaves.txt,Leaves of Grass,5.531950
1,2,melville-moby_dick.txt,Moby Dick,5.465767
2,3,milton-paradise.txt,Paradise Lost,4.993437
3,4,chesterton-brown.txt,The Wisdom of Father Brown,4.850803
4,5,chesterton-ball.txt,The Ball and the Cross,4.501512



## Consulta: relaciones familiares (family love marriage children home)


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,blake-poems.txt,Poems of William Blake,0.031692,"arth a little space, That we may learn to bear..."
1,2,austen-persuasion.txt,Persuasion,0.017773,"proved it by adding, for the information of hi..."
2,3,austen-sense.txt,Sense and Sensibility,0.014873,[Sense and Sensibility by Jane Austen 1811] CH...
3,4,whitman-leaves.txt,Leaves of Grass,0.014511,"ery ship that sails, sail you! Bear forth to t..."
4,5,austen-emma.txt,Emma,0.014285,"a Woodhouse, handsome, clever, and rich, with ..."


... (6 outputs left out)


## Consulta: naturaleza y poesía (nature earth flowers night poem)


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,blake-poems.txt,Poems of William Blake,0.028589,[Poems by William Blake 1789] SONGS OF INNOCEN...
1,2,milton-paradise.txt,Paradise Lost,0.025688,taught the chosen seed In the beginning how th...
2,3,whitman-leaves.txt,Leaves of Grass,0.017681,There to some group of mates the chants resumi...
3,4,shakespeare-macbeth.txt,The Tragedy of Macbeth,0.015441,"ie to be a Rebell, for to that The multiplying..."
4,5,shakespeare-hamlet.txt,The Tragedy of Hamlet,0.013480,had quiet Guard? Fran. Not a Mouse stirring Ba...


Top-5 BM25


,Posición,Identificador,Título,Puntuación BM25
0,1,whitman-leaves.txt,Leaves of Grass,2.304649
1,2,chesterton-ball.txt,The Ball and the Cross,1.984786
2,3,shakespeare-hamlet.txt,The Tragedy of Hamlet,1.780872
3,4,bryant-stories.txt,Stories by William Cullen Bryant,1.686553
4,5,chesterton-thursday.txt,The Man Who Was Thursday,1.685611
